# Procesamiento con spaCy

Se carga `es_core_news_lg` y se procesa el corpus con `nlp.pipe()`. Para la pregunta del trabajo se utilizan tres operaciones: POS, lematización y morfología.

La tabla principal compara la frecuencia absoluta y relativa de adjetivos (ADJ) y adverbios (ADV) por obra.

In [1]:
import json
from collections import Counter

import pandas as pd
import spacy


In [2]:
CORPUS_PATH = "corpus.jsonl"
MODEL = "es_core_news_lg"

corpus = pd.read_json(CORPUS_PATH, lines=True)

required = {"id", "titulo", "autor", "texto"}
missing = required - set(corpus.columns)
if missing:
    raise ValueError(f"Faltan columnas requeridas en corpus.jsonl: {sorted(missing)}")

corpus[["id", "titulo", "autor"]].head()

,id,titulo,autor
0,doc_01,Caramurú - Wikisource,Alejandro Magariños Cervantes
1,doc_02,Doña Luz - Wikisource,Juan Valera
2,doc_03,A flor de piel - Wikisource,Antonio de Hoyos y Vinent
3,doc_04,Don Segundo Sombra - Wikisource,AutorRicardo Güiraldes
4,doc_05,A fuego lento - Wikisource,Emilio Bobadilla


In [3]:
nlp = spacy.load(MODEL, disable=["parser", "ner"])
print(nlp.meta["name"], nlp.meta["version"])


core_news_lg 3.8.0


## 1. POS: adjetivos y adverbios

Se procesan los documentos de forma secuencial con `nlp.pipe()`. Se cuentan los tokens etiquetados como `ADJ` y `ADV` y se calculan sus frecuencias relativas sobre el total de tokens lingüísticos procesados.

In [4]:
pos_rows = []

texts = corpus["texto"].fillna("").tolist()
rows = corpus[["id", "titulo", "autor"]].to_dict("records")

for row, doc in zip(rows, nlp.pipe(texts, batch_size=1)):
    total = len(doc)
    counts = Counter(token.pos_ for token in doc)
    pos_rows.append({
        "id": row["id"],
        "titulo": row["titulo"],
        "autor": row["autor"],
        "tokens": total,
        "ADJ": counts["ADJ"],
        "ADV": counts["ADV"],
        "ADJ_rel": counts["ADJ"] / total if total else 0,
        "ADV_rel": counts["ADV"] / total if total else 0,
    })

tabla_pos = pd.DataFrame(pos_rows)
tabla_pos

,id,titulo,autor,tokens,ADJ,ADV,ADJ_rel,ADV_rel
0,doc_01,Caramurú - Wikisource,Alejandro Magariños Cervantes,57862,3793,2066,0.065553,0.035706
1,doc_02,Doña Luz - Wikisource,Juan Valera,69075,4138,3607,0.059906,0.052219
2,doc_03,A flor de piel - Wikisource,Antonio de Hoyos y Vinent,75054,6100,2695,0.081275,0.035907
3,doc_04,Don Segundo Sombra - Wikisource,AutorRicardo Güiraldes,74269,3806,2961,0.051246,0.039869
4,doc_05,A fuego lento - Wikisource,Emilio Bobadilla,78269,4762,2945,0.060841,0.037627
5,doc_06,Dulce dueño (Pardo Bazán) - Wikisource,Emilia Pardo Bazán,82544,5410,3367,0.065541,0.040790
6,doc_07,Crónica del reinado de Carlos IX - Wikisource,AutorProsper Mérimée,81288,4093,3445,0.050352,0.042380
7,doc_08,Doña Milagros - Wikisource,Emilia Pardo Bazán,81105,4794,3773,0.059109,0.046520
8,doc_09,De Cartago a Sagunto - Wikisource,Benito Pérez Galdós,78701,5136,2832,0.065260,0.035984
9,doc_10,Aurora roja - Wikisource,Pío Baroja,86994,4274,3460,0.049130,0.039773


## 2. Lemas

La lematización permite agrupar distintas formas flexionadas bajo una misma forma base. Para mantener la tabla enfocada en la pregunta, se muestran los lemas más frecuentes de ADJ y ADV por obra.

In [5]:
lemma_counts = Counter()

for row, doc in zip(rows, nlp.pipe(texts, batch_size=1)):
    for token in doc:
        if token.pos_ in {"ADJ", "ADV"} and not token.is_punct and not token.is_space:
            lemma_counts[(row["titulo"], token.pos_, token.lemma_.lower())] += 1

tabla_lemmas = (
    pd.DataFrame(
        [
            {"titulo": titulo, "pos": pos, "lemma": lemma, "frecuencia": freq}
            for (titulo, pos, lemma), freq in lemma_counts.items()
        ]
    )
    .sort_values(["titulo", "pos", "frecuencia"], ascending=[True, True, False])
    .groupby(["titulo", "pos"], group_keys=False)
    .head(20)
    .reset_index(drop=True)
    )
tabla_lemmas

,titulo,pos,lemma,frecuencia
0,A flor de piel - Wikisource,ADJ,viejo,68
1,A flor de piel - Wikisource,ADJ,negro,58
2,A flor de piel - Wikisource,ADJ,gran,58
3,A flor de piel - Wikisource,ADJ,grande,57
4,A flor de piel - Wikisource,ADJ,primero,57
...,...,...,...,...
475,Dulce dueño (Pardo Bazán) - Wikisource,ADV,antes,40
476,Dulce dueño (Pardo Bazán) - Wikisource,ADV,poco,37
477,Dulce dueño (Pardo Bazán) - Wikisource,ADV,nunca,37
478,Dulce dueño (Pardo Bazán) - Wikisource,ADV,acaso,32


## 3. Morfología

Se extraen rasgos morfológicos disponibles para ADJ y ADV. La tabla permite observar, por ejemplo, la distribución de género y número en los adjetivos.

In [6]:
morph_counts = Counter()

for row, doc in zip(rows, nlp.pipe(texts, batch_size=1)):
    for token in doc:
        if token.pos_ not in {"ADJ", "ADV"} or token.is_punct or token.is_space:
            continue

        morph = token.morph.to_dict()
        genero = morph.get("Gender", "—")
        numero = morph.get("Number", "—")
        grado = morph.get("Degree", "—")
        morph_counts[(row["titulo"], token.pos_, genero, numero, grado)] += 1

tabla_morfologia = pd.DataFrame(
    [
        {
            "titulo": titulo,
            "pos": pos,
            "genero": genero,
            "numero": numero,
            "grado": grado,
            "frecuencia": frecuencia,
        }
        for (titulo, pos, genero, numero, grado), frecuencia in morph_counts.items()
    ]
    ).sort_values(["titulo", "pos", "frecuencia"], ascending=[True, True, False]).reset_index(drop=True)
tabla_morfologia

,titulo,pos,genero,numero,grado,frecuencia
0,A flor de piel - Wikisource,ADJ,Fem,Sing,—,1694
1,A flor de piel - Wikisource,ADJ,Masc,Sing,—,1558
2,A flor de piel - Wikisource,ADJ,—,Sing,—,1082
3,A flor de piel - Wikisource,ADJ,Masc,Plur,—,692
4,A flor de piel - Wikisource,ADJ,Fem,Plur,—,537
...,...,...,...,...,...,...
172,Dulce dueño (Pardo Bazán) - Wikisource,ADJ,Fem,Sing,Sup,2
173,Dulce dueño (Pardo Bazán) - Wikisource,ADJ,Fem,Plur,Abs,1
174,Dulce dueño (Pardo Bazán) - Wikisource,ADJ,Masc,Plur,Abs,1
175,Dulce dueño (Pardo Bazán) - Wikisource,ADV,—,—,—,3092


## Evidencia de procesamiento

Las tres variables siguientes son las tablas pandas que se pueden presentar como evidencia del procesamiento con spaCy:

- `tabla_pos`: comparación de ADJ y ADV por obra.
- `tabla_lemmas`: lemas más frecuentes de ADJ y ADV.
- `tabla_morfologia`: rasgos morfológicos de ADJ y ADV.

In [ ]:
tabla_pos.head(12), tabla_lemmas.head(20), tabla_morfologia.head(20)